<a href="https://colab.research.google.com/github/SenTier1107/Appprogramming_2026/blob/main/FastAPI_SyncAsync/sync_async_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  동기(Sync) vs  비동기(Async) 완전 정복

---

##  학습 목표

이 노트북을 마치면 다음을 할 수 있습니다:

1. 동기(Sync)와 비동기(Async)의 개념 차이를 설명할 수 있다
2. `asyncio`, `aiohttp`를 사용한 비동기 코드를 작성할 수 있다
3. `async def`, `await`, `asyncio.gather()` 키워드를 올바르게 사용할 수 있다
4. 실제 파일 다운로드 예제로 성능 차이를 직접 측정할 수 있다

---

##  개념: 동기 ,비동기란?

###  커피샵 비유로 이해하기

| 상황 | 동기(Sync) | 비동기(Async) |
|------|-----------|---------------|
| 친구한테 커피 부탁 | 커피 받을 때까지 그 자리에서 기다린다 | 커피 기다리는 동안 핸드폰 보거나 다른 일을 한다 |
| 특징 | 순서 보장, 코드 단순 | 효율적, 코드가 약간 복잡 |

###  실행 흐름 비교

```
동기:  [=작업1=][=작업2=][=작업3=]  → 총 시간 = 합산

비동기: [=작업1=]
        [==작업2==]
        [=작업3=]
        → 총 시간 = 가장 긴 작업 1개
```

###   비동기를 쓰는 경우
- **네트워크 요청** (웹 크롤링, API 호출, 파일 다운로드)
- **파일 I/O** (대용량 파일 읽기/쓰기)
- 공통점: **기다리는 시간(I/O 대기)**이 많은 작업!

###  비동기가 오히려 비효율인 경우
- CPU 연산 위주 작업 (행렬 계산, 이미지 처리)
- 순서가 반드시 보장되어야 하는 작업

---

## 0️. 공통 설정 및 준비

아래 셀을 먼저 실행하세요.

In [1]:
!pip install aiohttp requests beautifulsoup4 -q
print("✅ 설치 완료")

✅ 설치 완료


In [2]:
import os
import time
import requests
from bs4 import BeautifulSoup
import aiohttp
import asyncio

print("✅ 라이브러리 임포트 완료")

✅ 라이브러리 임포트 완료


In [5]:
def get_all_csv_urls(base_url):
    """주어진 base_url 페이지에서 .csv 링크를 모두 수집하여 반환"""
    response = requests.get(base_url)
    if response.status_code != 200:
        return []
    soup = BeautifulSoup(response.text, "html.parser")
    return [
        base_url + link.get("href")
        for link in soup.find_all("a")
        if link.get("href", "").endswith(".csv")
    ]

BASE_URL = "https://people.sc.fsu.edu/~jburkardt/data/csv/"
file_urls = get_all_csv_urls(BASE_URL)

print(f"총 {len(file_urls)}개의 CSV 파일 URL을 찾았습니다.")
for url in file_urls[:5]:
    print(" -", url)
if len(file_urls) > 5:
    print(f" ... 외 {len(file_urls)-5}개")

총 33개의 CSV 파일 URL을 찾았습니다.
 - https://people.sc.fsu.edu/~jburkardt/data/csv/addresses.csv
 - https://people.sc.fsu.edu/~jburkardt/data/csv/airtravel.csv
 - https://people.sc.fsu.edu/~jburkardt/data/csv/biostats.csv
 - https://people.sc.fsu.edu/~jburkardt/data/csv/cities.csv
 - https://people.sc.fsu.edu/~jburkardt/data/csv/crash_catalonia.csv
 ... 외 28개


In [4]:
def clean_csv_files():
    """현재 디렉토리의 모든 .csv 파일을 삭제 (실습 후 정리용)"""
    deleted = []
    for file_name in os.listdir("./"):
        if file_name.endswith(".csv"):
            file_path = os.path.join("./", file_name)
            try:
                os.remove(file_path)
                deleted.append(file_name)
            except OSError as e:
                print(f"삭제 실패: {file_path} - {e.strerror}")
    print(f"{len(deleted)}개 CSV 파일 삭제 완료.")

print("✅ clean_csv_files() 함수 정의 완료")

✅ clean_csv_files() 함수 정의 완료


---

## 1️. 동기 방식으로 파일 다운로드

###  개념 설명

동기 방식은 코드가 **위에서 아래로 순서대로** 실행됩니다.

```
파일1 시작 → 파일1 완료 → 파일2 시작 → 파일2 완료 → ...
```

- `requests.get(url)` 호출 시 응답이 올 때까지 프로그램이 **멈춤(Blocking)**
- 코드가 단순하고 이해하기 쉬움

>  **실행 후 관찰:** `[1]` → `[2]` → `[3]` 순서로 완료 메시지가 출력됩니다!

In [6]:
def download_file_sync(url, save_path, index):
    """단일 파일을 동기 방식으로 다운로드"""
    print(f"[{index:2d}] 다운로드 시작: {os.path.basename(url)}")
    # requests.get()은 응답이 올 때까지 프로그램을 블로킹(정지)시킵니다
    response = requests.get(url)
    if response.status_code == 200:
        with open(save_path, "wb") as f:
            f.write(response.content)
        print(f"[{index:2d}] ✅ 완료: {save_path}")
        return save_path
    else:
        print(f"[{index:2d}] ❌ 실패: {url} (상태코드: {response.status_code})")
        return None


def download_all_sync():
    """모든 파일을 동기 방식으로 순차 다운로드"""
    results = []
    for i, url in enumerate(file_urls, start=1):
        save_path = f"./{os.path.basename(url)}"
        result = download_file_sync(url, save_path, i)
        if result:
            results.append(os.path.basename(result))
    return results


print("=" * 50)
print("동기 방식 다운로드 시작")
print("=" * 50)
t1 = time.time()
sync_results = download_all_sync()
t2 = time.time()

sync_time = t2 - t1
print(f"\n⏱ 동기 방식 완료: {sync_time:.2f}초 소요")
print(f"📁 다운로드된 파일 목록 ({len(sync_results)}개):")
for idx, fname in enumerate(sync_results, start=1):
    print(f"  {idx:2d}. {fname}")

동기 방식 다운로드 시작
[ 1] 다운로드 시작: addresses.csv
[ 1] ✅ 완료: ./addresses.csv
[ 2] 다운로드 시작: airtravel.csv
[ 2] ✅ 완료: ./airtravel.csv
[ 3] 다운로드 시작: biostats.csv
[ 3] ✅ 완료: ./biostats.csv
[ 4] 다운로드 시작: cities.csv
[ 4] ✅ 완료: ./cities.csv
[ 5] 다운로드 시작: crash_catalonia.csv
[ 5] ✅ 완료: ./crash_catalonia.csv
[ 6] 다운로드 시작: deniro.csv
[ 6] ✅ 완료: ./deniro.csv
[ 7] 다운로드 시작: example.csv
[ 7] ✅ 완료: ./example.csv
[ 8] 다운로드 시작: faithful.csv
[ 8] ✅ 완료: ./faithful.csv
[ 9] 다운로드 시작: ford_escort.csv
[ 9] ✅ 완료: ./ford_escort.csv
[10] 다운로드 시작: freshman_kgs.csv
[10] ✅ 완료: ./freshman_kgs.csv
[11] 다운로드 시작: freshman_lbs.csv
[11] ✅ 완료: ./freshman_lbs.csv
[12] 다운로드 시작: grades.csv
[12] ✅ 완료: ./grades.csv
[13] 다운로드 시작: homes.csv
[13] ✅ 완료: ./homes.csv
[14] 다운로드 시작: hooke.csv
[14] ✅ 완료: ./hooke.csv
[15] 다운로드 시작: hurricanes.csv
[15] ✅ 완료: ./hurricanes.csv
[16] 다운로드 시작: hw_200.csv
[16] ✅ 완료: ./hw_200.csv
[17] 다운로드 시작: hw_25000.csv
[17] ✅ 완료: ./hw_25000.csv
[18] 다운로드 시작: lead_shot.csv
[18] ✅ 완료: ./lead_shot.csv
[19] 다운로드 시작: le

In [7]:
clean_csv_files()

33개 CSV 파일 삭제 완료.


---

## 2️. 비동기 방식으로 파일 다운로드

###  핵심 라이브러리 설명

#### `asyncio` — 비동기 이벤트 루프 관리자

```
비유: 여러 요리를 동시에 하는 주방장
  국이 끓는 동안 → 다른 재료를 손질
  밥이 뜸들이는 동안 → 반찬을 준비
```

| 키워드 | 의미 |
|--------|------|
| `async def` | 비동기 함수 정의. 이벤트 루프를 통해야 실행됨 |
| `await` | 이 작업이 끝날 때까지 대기, 그 동안 다른 작업 가능 |
| `asyncio.gather()` | 여러 코루틴을 동시에 실행하고 결과 수집 |
| `asyncio.run()` | 일반 .py 스크립트에서 비동기 코드를 실행하는 진입점 |

#### `aiohttp` — 비동기 HTTP 통신 라이브러리

```python
# 동기: 한 번에 하나씩, 응답 올 때까지 대기
response = requests.get(url)

# 비동기: 수백 개를 거의 동시에, 오는 대로 처리
async with session.get(url) as response:
    text = await response.text()
```

>  **실행 후 관찰:**
> - `다운로드 시작` 메시지가 `[1]~[N]`이 거의 동시에 출력됩니다
> - `완료` 메시지 순서가 요청 순서와 다를 수 있습니다
> - 전체 소요 시간이 동기보다 훨씬 짧습니다!

In [8]:
async def download_file_async(session, url, save_path, index):
    """단일 파일을 비동기 방식으로 다운로드"""
    print(f"[{index:2d}] 다운로드 시작: {os.path.basename(url)}")
    async with session.get(url) as response:
        if response.status == 200:
            content = await response.read()  # await: 데이터 읽기 완료까지 대기
            with open(save_path, "wb") as f:
                f.write(content)
            print(f"[{index:2d}] ✅ 완료: {save_path}")
            return save_path
        else:
            print(f"[{index:2d}] ❌ 실패: {url} (상태코드: {response.status})")
            return None


async def download_all_async():
    """모든 파일을 비동기 방식으로 동시 다운로드"""
    async with aiohttp.ClientSession() as session:
        tasks = [
            download_file_async(session, url, f"./{os.path.basename(url)}", i)
            for i, url in enumerate(file_urls, start=1)
        ]
        results = await asyncio.gather(*tasks)
    return [os.path.basename(r) for r in results if r]


print("=" * 50)
print("비동기 방식 다운로드 시작")
print("=" * 50)
t1 = time.time()
async_results = await download_all_async()   # 일반 스크립트: asyncio.run(download_all_async())
t2 = time.time()

async_time = t2 - t1
print(f"\n⏱ 비동기 방식 완료: {async_time:.2f}초 소요")
print(f"📁 다운로드된 파일 목록 ({len(async_results)}개):")
for idx, fname in enumerate(async_results, start=1):
    print(f"  {idx:2d}. {fname}")

비동기 방식 다운로드 시작
[ 1] 다운로드 시작: addresses.csv
[ 2] 다운로드 시작: airtravel.csv
[ 3] 다운로드 시작: biostats.csv
[ 4] 다운로드 시작: cities.csv
[ 5] 다운로드 시작: crash_catalonia.csv
[ 6] 다운로드 시작: deniro.csv
[ 7] 다운로드 시작: example.csv
[ 8] 다운로드 시작: faithful.csv
[ 9] 다운로드 시작: ford_escort.csv
[10] 다운로드 시작: freshman_kgs.csv
[11] 다운로드 시작: freshman_lbs.csv
[12] 다운로드 시작: grades.csv
[13] 다운로드 시작: homes.csv
[14] 다운로드 시작: hooke.csv
[15] 다운로드 시작: hurricanes.csv
[16] 다운로드 시작: hw_200.csv
[17] 다운로드 시작: hw_25000.csv
[18] 다운로드 시작: lead_shot.csv
[19] 다운로드 시작: letter_frequency.csv
[20] 다운로드 시작: mlb_players.csv
[21] 다운로드 시작: mlb_teams_2012.csv
[22] 다운로드 시작: news_decline.csv
[23] 다운로드 시작: nile.csv
[24] 다운로드 시작: oscar_age_female.csv
[25] 다운로드 시작: oscar_age_male.csv
[26] 다운로드 시작: snakes_count_10.csv
[27] 다운로드 시작: snakes_count_100.csv
[28] 다운로드 시작: snakes_count_1000.csv
[29] 다운로드 시작: snakes_count_10000.csv
[30] 다운로드 시작: tally_cab.csv
[31] 다운로드 시작: taxables.csv
[32] 다운로드 시작: trees.csv
[33] 다운로드 시작: zillow.csv
[27] ✅ 완료: ./snakes_count

In [9]:
clean_csv_files()

33개 CSV 파일 삭제 완료.


---

## 3️. 성능 비교 결과 요약

In [24]:
print("=" * 50)
print("   성능 비교 결과")
print("=" * 50)
print(f"   동기 방식:  {sync_time:.2f}초")
print(f"   비동기 방식: {async_time:.2f}초")
print(f"   속도 향상:   {sync_time / async_time:.1f}배 빠름")
print("=" * 50)

   성능 비교 결과
   동기 방식:  5.50초
   비동기 방식: 0.59초
   속도 향상:   9.3배 빠름


###  동기 vs 비동기 핵심 차이 정리

| 항목 | 동기 (Sync) | 비동기 (Async) |
|------|------------|----------------|
| 처리 방식 | 순차 (하나씩) | 동시 (한꺼번에) |
| 완료 순서 | 항상 요청 순서와 동일 | 응답이 빠른 것부터 완료 |
| 소요 시간 | 파일 수 × 개당 시간 | ≈ 가장 느린 파일 1개의 시간 |
| 코드 복잡도 | 단순 | `async/await` 필요 |
| 적합한 상황 | 순서 보장, 단순 작업 | 네트워크 I/O, 대량 요청 |

---

## 4️. `async` / `await` / `asyncio` 핵심 개념 실습

`await`를 만나면:
1. 현재 함수가 "잠깐 대기" 상태로 전환
2. 이벤트 루프가 다른 준비된 코루틴을 실행
3. `await` 작업이 완료되면 다시 이 함수로 돌아옴

>  프로그램이 멈추는 게 아니라, "나는 기다릴게, 그동안 다른 일 해"! 라는 개념이다.

In [11]:
async def task(name, delay):
    print(f"[{name}] 시작")
    await asyncio.sleep(delay)  # delay초 기다리는 동안 다른 task 실행됨
    print(f"[{name}] 완료 ({delay}초 후)")

print("=== 비동기: 9개 작업을 동시에 실행 ===")
t1 = time.time()

await asyncio.gather(
    task("A", 3),
    task("B", 1),
    task("C", 2),
    task("D", 3),
    task("E", 1),
    task("F", 2),
    task("G", 3),
    task("H", 1),
    task("I", 2),
)
t2 = time.time()
print(f"\n총 소요 시간: {t2 - t1:.1f}초 (순차라면 18초 소요됐을 것)")

=== 비동기: 9개 작업을 동시에 실행 ===
[A] 시작
[B] 시작
[C] 시작
[D] 시작
[E] 시작
[F] 시작
[G] 시작
[H] 시작
[I] 시작
[B] 완료 (1초 후)
[E] 완료 (1초 후)
[H] 완료 (1초 후)
[C] 완료 (2초 후)
[F] 완료 (2초 후)
[I] 완료 (2초 후)
[A] 완료 (3초 후)
[D] 완료 (3초 후)
[G] 완료 (3초 후)

총 소요 시간: 3.0초 (순차라면 18초 소요됐을 것)


###   (왜 3초인가?)

| 방식 | 계산 방식 | 예상 소요 시간 |
|------|----------|----------------|
| 순차 실행 (Synchronous) | 3+1+2+3+1+2+3+1+2 | **18초** |
| 비동기 실행 (Asynchronous) | max(3,1,2,3,1,2,3,1,2) | **3초** |

비동기는 모든 작업이 **동시에** 시작되므로, 전체 시간 = 가장 긴 작업 1개의 시간!

---

## 5️. aiohttp로 웹 페이지 내용 가져오기

`requests.get()` ↔ `async with session.get()` 대응 관계를 확인하세요.

- **동기 (fetch_sync)**: `requests.get()`으로 URL에 요청, 응답 올 때까지 블로킹
- **비동기 (fetch_async)**: `aiohttp`로 요청, 응답 대기 중 다른 작업 병행 가능

In [12]:
import urllib3
urllib3.disable_warnings()

# 동기 방식
def fetch_sync(url):
    response = requests.get(url, verify=False)
    return response.text

# 비동기 방식
async def fetch_async(url):
    async with aiohttp.ClientSession(connector=aiohttp.TCPConnector(ssl=False)) as session:
        async with session.get(url) as response:
            return await response.text()


url = "https://example.com"

html_sync = fetch_sync(url)
print("[동기] 결과 (앞 200자):")
print(html_sync[:200])
print()

html_async = await fetch_async(url)  # 일반 스크립트: asyncio.run(fetch_async(url))
print("[비동기] 결과 (앞 200자):")
print(html_async[:200])

[동기] 결과 (앞 200자):
<!doctype html><html lang="en"><head><title>Example Domain</title><meta name="viewport" content="width=device-width, initial-scale=1"><style>body{background:#eee;width:60vw;margin:15vh auto;font-famil

[비동기] 결과 (앞 200자):
<!doctype html><html lang="en"><head><title>Example Domain</title><meta name="viewport" content="width=device-width, initial-scale=1"><style>body{background:#eee;width:60vw;margin:15vh auto;font-famil


---

## 여러 URL 동시에 가져오기

**목표:** `asyncio.gather()`를 사용해 여러 웹페이지를 동시에 가져오는 함수를 완성하세요.

```python
#  구조
async def fetch_page(session, url):
    # aiohttp session으로 url 요청 → 상태코드 + 길이 반환
    ...

async def fetch_all(urls):
    async with aiohttp.ClientSession(...) as session:
        tasks = [fetch_page(session, url) for url in urls]
        results = await asyncio.gather(*tasks)
    return results
```

In [14]:
async def fetch_page_answer(session, url):
    try:
        async with session.get(url, ssl=False) as response:
            content = await response.text()
            return {"url": url, "status": response.status, "length": len(content)}
    except Exception as e:
        return {"url": url, "status": "error", "length": 0}


async def fetch_all_answer(urls):
    async with aiohttp.ClientSession(connector=aiohttp.TCPConnector(ssl=False)) as session:
        tasks = [fetch_page_answer(session, url) for url in urls]
        results = await asyncio.gather(*tasks)
    return results


print("=" * 55)
print("여러 URL 동시 요청 결과")
print("=" * 55)
t1 = time.time()
results = await fetch_all_answer(urls_to_fetch)
t2 = time.time()

for r in results:
    print(f"  [{r['status']}] {r['url']} → {r['length']:,}자")
print(f"\n⏱ 총 소요 시간: {t2-t1:.2f}초 ({len(urls_to_fetch)}개 동시 요청)")

여러 URL 동시 요청 결과
  [200] https://example.com → 528자
  [200] https://jsonplaceholder.typicode.com/posts/1 → 292자
  [200] https://jsonplaceholder.typicode.com/posts/2 → 278자
  [200] https://jsonplaceholder.typicode.com/posts/3 → 283자
  [200] https://jsonplaceholder.typicode.com/posts/4 → 270자

⏱ 총 소요 시간: 0.09초 (5개 동시 요청)


---

##동기 vs 비동기 직접 비교

**목표:** 같은 20개 URL을 동기·비동기로 각각 가져오고 시간을 비교하세요.

In [15]:
test_urls = [f"https://jsonplaceholder.typicode.com/posts/{i}" for i in range(1, 21)]
print(f"테스트 URL 수: {len(test_urls)}개")

# 동기 방식
def fetch_all_sync_test(urls):
    results = []
    for url in urls:
        r = requests.get(url)
        results.append(len(r.text))
    return results

print("\n[동기 방식 시작]")
t1 = time.time()
fetch_all_sync_test(test_urls)
sync_t = time.time() - t1
print(f"동기 완료: {sync_t:.2f}초")

# 비동기 방식
async def fetch_all_async_test(urls):
    async with aiohttp.ClientSession(connector=aiohttp.TCPConnector(ssl=False)) as session:
        async def get_len(url):
            async with session.get(url) as r:
                text = await r.text()
                return len(text)
        return await asyncio.gather(*[get_len(url) for url in urls])

print("[비동기 방식 시작]")
t1 = time.time()
await fetch_all_async_test(test_urls)
async_t = time.time() - t1
print(f"비동기 완료: {async_t:.2f}초")

print("\n" + "=" * 40)
print(f"  🐢 동기:  {sync_t:.2f}초")
print(f"  🚀 비동기: {async_t:.2f}초")
print(f"  ⚡ {sync_t/async_t:.1f}배 빠름")
print("=" * 40)

테스트 URL 수: 20개

[동기 방식 시작]
동기 완료: 1.67초
[비동기 방식 시작]
비동기 완료: 0.14초

  🐢 동기:  1.67초
  🚀 비동기: 0.14초
  ⚡ 12.2배 빠름


---

##  동시성 제한 (Semaphore)

###  무제한 동시 요청의 위험성

비동기로 수천 개의 요청을 한번에 보내면 서버 과부하 또는 차단 위험이 있습니다.

**해결책: `asyncio.Semaphore(n)`으로 동시 실행 수를 n개로 제한**

```
Semaphore = 신호등
  녹색 (획득 가능): 진행
  적색 (최대 수 도달): 대기
```

In [16]:
async def download_with_limit(urls, max_concurrent=5):
    """최대 max_concurrent개만 동시에 요청하는 비동기 다운로드"""
    semaphore = asyncio.Semaphore(max_concurrent)

    async def fetch_with_sem(session, url, idx):
        async with semaphore:  # 최대 수 초과 시 여기서 대기
            async with session.get(url) as response:
                content = await response.text()
                print(f"  [{idx:2d}] 완료 (동시 제한: {max_concurrent}개)")
                return len(content)

    async with aiohttp.ClientSession(connector=aiohttp.TCPConnector(ssl=False)) as session:
        tasks = [fetch_with_sem(session, url, i) for i, url in enumerate(urls, 1)]
        return await asyncio.gather(*tasks)


test_urls_20 = [f"https://jsonplaceholder.typicode.com/posts/{i}" for i in range(1, 21)]

print("=" * 50)
print(f"동시 제한 5개로 {len(test_urls_20)}개 URL 요청")
print("=" * 50)
t1 = time.time()
results = await download_with_limit(test_urls_20, max_concurrent=5)
t2 = time.time()
print(f"\n⏱ 완료: {t2-t1:.2f}초 (총 {len(results)}개 처리)")

동시 제한 5개로 20개 URL 요청
  [ 3] 완료 (동시 제한: 5개)
  [ 1] 완료 (동시 제한: 5개)
  [ 4] 완료 (동시 제한: 5개)
  [ 5] 완료 (동시 제한: 5개)
  [ 2] 완료 (동시 제한: 5개)
  [ 6] 완료 (동시 제한: 5개)
  [ 7] 완료 (동시 제한: 5개)
  [ 8] 완료 (동시 제한: 5개)
  [10] 완료 (동시 제한: 5개)
  [ 9] 완료 (동시 제한: 5개)
  [11] 완료 (동시 제한: 5개)
  [12] 완료 (동시 제한: 5개)
  [14] 완료 (동시 제한: 5개)
  [13] 완료 (동시 제한: 5개)
  [15] 완료 (동시 제한: 5개)
  [16] 완료 (동시 제한: 5개)
  [18] 완료 (동시 제한: 5개)
  [17] 완료 (동시 제한: 5개)
  [19] 완료 (동시 제한: 5개)
  [20] 완료 (동시 제한: 5개)

⏱ 완료: 0.14초 (총 20개 처리)


---

## 6️. Jupyter/Colab vs 일반 스크립트 차이점

| 환경 | 실행 방법 | 이유 |
|------|----------|------|
| Jupyter / Colab | `await my_func()` | 이미 이벤트 루프가 실행 중 |
| 일반 .py 스크립트 | `asyncio.run(my_func())` | 이벤트 루프를 직접 생성해야 함 |

```python
# Jupyter/Colab
result = await my_async_function()          # ✅

# 일반 스크립트
result = asyncio.run(my_async_function())   # ✅
```

In [17]:
# 현재 이벤트 루프 확인
loop = asyncio.get_event_loop()
print(f"현재 이벤트 루프: {loop}")
print(f"이벤트 루프 실행 중: {loop.is_running()}")

현재 이벤트 루프: <_UnixSelectorEventLoop running=True closed=False debug=False>
이벤트 루프 실행 중: True


---

## 학습 내용 요약

### 핵심 개념 체크리스트

- [ ] **동기(Sync)**: 순차 실행, 코드 단순, I/O 대기 중 CPU 낭비
- [ ] **비동기(Async)**: 동시 실행, I/O 효율적, `async/await` 문법 필요
- [ ] **`async def`**: 코루틴 함수 정의 (이벤트 루프를 통해 실행됨)
- [ ] **`await`**: 비동기 작업 대기 (그 동안 다른 코루틴 실행 가능)
- [ ] **`asyncio.gather()`**: 여러 코루틴 동시 실행 + 결과 수집
- [ ] **`asyncio.Semaphore(n)`**: 동시 실행 수를 n개로 제한
- [ ] **`aiohttp`**: 비동기 HTTP 요청 라이브러리 (`requests`의 비동기 버전)
- [ ] **Jupyter vs .py**: `await` 직접 사용 vs `asyncio.run()` 사용
